# 03 · MICE Imputation
Multiple imputation by chained equations (MICE) on the five ordinal SED
variables across the full AoU cohort (~413k persons).

| Variable | MICE method | Rationale |
|----------|------------|----------|
| Income | `polr` | Ordered logistic (9 levels) |
| Education | `polr` | Ordered logistic (8 levels) |
| Housing | `polr` | Ordered logistic (3 levels) |
| Employment | `polr` | Ordered logistic (4 levels) |
| Insurance | `logreg` | Binary logistic |

Outputs 5 completed datasets (`imputed_ds{1..5}_trial2.tsv`) to GCS.


---
### Citation

This pipeline implements the iSDI construction described in:

> **Reference:** Gupta S, Lam V, Jordan IK, Mariño-Ramírez L. *A composite socioeconomic deprivation index from All of Us survey data: associations with health outcomes and disparities.* medRxiv 2024.10.04.24314904. PMID: [39802760](https://pubmed.ncbi.nlm.nih.gov/39802760/). doi: [10.1101/2024.10.04.24314904](https://doi.org/10.1101/2024.10.04.24314904)


In [ ]:
# 03_sed_mice_imputation.R
# -----------------------------------------------------------------------
# Multiple imputation of the five ordinal SED variables using MICE.
# Runs on the full AoU cohort (~413k persons).
#
# Input:  GCS  $WORKSPACE_BUCKET/data/sdi_var_df_new_order_with_spearman.csv
#         (person_id, Income, Education, Housing, Employment, Insurance)
# Output: GCS  $WORKSPACE_BUCKET/data/imputed_ds{1..5}_trial2.tsv
#         (5 completed datasets, one per MICE chain)
#
# MICE method choices:
#   Income, Education, Employment  →  "polr"   (ordered logistic, polr)
#   Housing                        →  "polr"   (3 levels)
#   Insurance                      →  "logreg" (binary logistic)
#
# BUG FIXES vs original notebook:
#   1. mean() imputation used without na.rm=TRUE → fixed
#   2. mice() argument spelled maxint; correct spelling is maxit → fixed
#   3. subset_df referenced after session reset (object not found) → all
#      code consolidated into a single linear script
#   4. combined_data column name typo "Isurance" → fixed to "Insurance"
#   5. Exploration subset: sample() failed because df was NULL at that point
#      (first read_csv path was wrong); corrected to use the correct file
# -----------------------------------------------------------------------

library(tidyverse)
library(mice)
library(readr)

# -----------------------------------------------------------------------
# 1. Load data from GCS
# -----------------------------------------------------------------------

my_bucket <- Sys.getenv("WORKSPACE_BUCKET")
local_path <- "sdi_var_df_new_order_with_spearman.csv"

system(paste0("gsutil cp ", my_bucket, "/data/", local_path, " ."), intern = TRUE)

df <- read_csv(local_path, show_col_types = FALSE)
cat(sprintf("Loaded: %d rows × %d cols\n", nrow(df), ncol(df)))
print(colSums(is.na(df)))

# -----------------------------------------------------------------------
# 2. Cast to ordered factors (required by polr/logreg methods)
# -----------------------------------------------------------------------

sed_vars <- c("Income", "Education", "Housing", "Employment", "Insurance")

df$Income     <- factor(df$Income,     ordered = TRUE)
df$Education  <- factor(df$Education,  ordered = TRUE)
df$Housing    <- factor(df$Housing,    ordered = TRUE)
df$Employment <- factor(df$Employment, ordered = TRUE)
df$Insurance  <- factor(df$Insurance,  ordered = TRUE)  # binary but keep consistent

# -----------------------------------------------------------------------
# 3. Pilot run on 10k subsample to check convergence
# -----------------------------------------------------------------------

set.seed(123)
subset_df <- df[sample(nrow(df), 10000), ]

pilot <- mice(
  subset_df[, sed_vars],
  m       = 5,
  method  = c("polr", "polr", "polr", "polr", "logreg"),
  maxit   = 20,                    # BUG FIX #2: was misspelled "maxint"
  seed    = 123,
  printFlag = FALSE
)
cat("\nPilot imputation summary:\n")
print(summary(pilot))

# Spearman correlation on one completed pilot dataset
d_pilot <- complete(pilot, 1)
d_pilot[] <- lapply(d_pilot, as.numeric)
c_pilot <- cor(d_pilot, method = "spearman", use = "pairwise.complete.obs")
cat("\nPilot Spearman correlation (imputed):\n")
print(round(c_pilot, 3))

# -----------------------------------------------------------------------
# 4. Full-cohort imputation (optional: add fake NAs for validation first)
# -----------------------------------------------------------------------

# Uncomment the block below to inject known-missing values for imputation QC.
# In production, skip this and impute on the real missingness pattern.
#
# df$Income[sample(nrow(df), 10000)]    <- NA
# df$Education[sample(nrow(df), 1300)]  <- NA
# df$Housing[sample(nrow(df), 6000)]    <- NA
# df$Employment[sample(nrow(df), 3000)] <- NA
# df$Insurance[sample(nrow(df), 700)]   <- NA
# cat("NA counts after artificial injection:\n")
# print(colSums(is.na(df)))

imputed_data <- mice(
  df[, sed_vars],
  m       = 5,
  method  = c("polr", "polr", "polr", "polr", "logreg"),
  maxit   = 20,
  seed    = 123,
  printFlag = TRUE
)

# -----------------------------------------------------------------------
# 5. Extract completed datasets and upload to GCS
# -----------------------------------------------------------------------

upload_imputed <- function(imp_obj, chain_idx, bucket) {
  d <- complete(imp_obj, chain_idx)
  fname <- sprintf("imputed_ds%d_trial2.tsv", chain_idx)
  write_excel_csv(d, fname)
  system(paste0("gsutil cp ./", fname, " ", bucket, "/data/"), intern = TRUE)
  message(sprintf("Uploaded: %s/data/%s", bucket, fname))
  invisible(d)
}

final_ds <- lapply(1:5, function(i) upload_imputed(imputed_data, i, my_bucket))

# -----------------------------------------------------------------------
# 6. Post-imputation correlation (chain 1)
# -----------------------------------------------------------------------

d1 <- final_ds[[1]]
d1[] <- lapply(d1, as.numeric)
c_imp <- cor(d1, method = "spearman", use = "pairwise.complete.obs")
cat("\nPost-imputation Spearman correlation (chain 1):\n")
print(round(c_imp, 3))

# -----------------------------------------------------------------------
# 7. Diagnostic: compare original vs imputed side-by-side
# -----------------------------------------------------------------------

# BUG FIX #4: typo "Isurance" corrected
combined_data <- cbind(
  df[, sed_vars] |> setNames(paste0(sed_vars, ".org")),
  d1             |> setNames(paste0(sed_vars, ".imp"))
)
head(combined_data)


> **Note on runtime:** The full-cohort MICE call (~413k × 5 variables,
> 5 chains × 20 iterations) takes ~2–3 hours on a standard AoU Jupyter
> environment. Run the pilot subsample block first to verify convergence.
